# 07 — Parameters and equations

Parameters are named values inside a part. Features can reference them
by name, so changing one parameter propagates through every dimension
that uses it.

## What you'll do
1. Build a small extrusion whose depth is a named parameter
2. List existing parameters
3. Drive the depth by changing the parameter
4. Add a free-standing parameter `A`, then a parameter `B = A * 2`
5. Push `A = 10`, watch `B` follow

**Prereq:** open a fresh empty part in Alibre.

## Setup: a parametric box

In [ ]:
from alibrex import (
    CurrentPart,
    ADDirectionType,
    ADParameterType,
    ADPartFeatureEndCondition,
)

part = CurrentPart()

xy = part.DesignPlanes.Item(0)
sk = part.Sketches.AddSketch(None, xy, "Base")
figs = sk.Figures
figs.AddLine(0.0, 0.0, 4.0, 0.0)
figs.AddLine(4.0, 0.0, 4.0, 2.0)
figs.AddLine(4.0, 2.0, 0.0, 2.0)
figs.AddLine(0.0, 2.0, 0.0, 0.0)

part.Features.AddExtrudedBoss(
    sk, 1.0, ADPartFeatureEndCondition.AD_TO_DEPTH,
    None, None, 0.0,
    ADDirectionType.AD_ALONG_NORMAL, None, None, False,
    None, False,
    "Box", "BoxDepth", "",     # <-- the depth becomes a parameter named "BoxDepth"
)

## List every parameter on the part

In [ ]:
params = part.Parameters
for i in range(params.Count):
    p = params.Item(i)
    eq = f"  =  {p.Equation}" if p.Equation else ""
    print(f"  {p.Name:20s} = {p.Value:8.4f}{eq}")

## Find one by name

There's no direct lookup; we iterate.

In [ ]:
depth = None
for i in range(params.Count):
    p = params.Item(i)
    if p.Name == "BoxDepth":
        depth = p
        break
assert depth is not None, "BoxDepth parameter not found on the part"
depth.Value

## Change `BoxDepth` from 1.0 to 2.5 cm

Parameter mutations run inside a transaction so the part regenerates
consistently.

In [ ]:
params.OpenParameterTransaction()
depth.Value = 2.5
params.CloseParameterTransaction()
part.RegenerateAll()
depth.Value

## Add a free-standing parameter `A`

In [ ]:
a = params.NewParameter("A", ADParameterType.AD_DISTANCE)
params.OpenParameterTransaction()
a.Value = 3.0
params.CloseParameterTransaction()
a.Value

## Add parameter `B` driven by an equation `A * 2`

In [ ]:
b = params.NewParameter("B", ADParameterType.AD_DISTANCE)
params.OpenParameterTransaction()
b.Equation = "A * 2"
params.CloseParameterTransaction()
part.RegenerateAll()
b.Value

## Push `A` to 10 — `B` follows

In [ ]:
params.OpenParameterTransaction()
a.Value = 10.0
params.CloseParameterTransaction()
part.RegenerateAll()
a.Value, b.Value

## Final dump

In [ ]:
for i in range(params.Count):
    p = params.Item(i)
    eq = f"  =  {p.Equation}" if p.Equation else ""
    print(f"  {p.Name:20s} = {p.Value:8.4f}{eq}")